# LC 199 — Binary Tree Right Side View
**Difficulty:** Medium | **Category:** BFS on Trees
**Pattern:** Level-by-Level BFS — Last Node Per Level

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Run the same level-order BFS as
LC 102. The only difference: instead of collecting every node in
a level, you take only the <em>last</em> node's value.
That last node is what a viewer standing to the right of the
tree would see at that depth.
</div>

## Official Problem Statement

Given the `root` of a binary tree, imagine yourself standing on
the **right side** of it. Return *the values of the nodes you can
see ordered from top to bottom*.

**Constraints:**
- The number of nodes in the tree is in the range `[0, 100]`.
- `-100 <= Node.val <= 100`

## What This Is Actually Asking

Stand to the right of the tree and look left.
At each level (depth), you can only see the rightmost node —
all other nodes at that level are hidden behind it.
Collect those rightmost nodes from the root's level down to the
deepest level.
Return their values as a flat list, top to bottom.

## Walk Through an Example by Hand

Tree: `[1, 2, 3, None, 5, None, 4]`

```
       1
      / \
     2   3
      \    \
       5    4
```

**Level 0:** queue=[1], level_size=1
- Pop 1 → level=[1], last=1
- Add children: 2, 3
- right_view = [1]

**Level 1:** queue=[2,3], level_size=2
- Pop 2 → level=[2] (add child 5)
- Pop 3 → level=[2,3] (add child 4), last=3
- right_view = [1, 3]

**Level 2:** queue=[5,4], level_size=2
- Pop 5 → level=[5]
- Pop 4 → level=[5,4], last=4
- right_view = [1, 3, 4]

## The Picture

```
        1           <- visible: 1
       / \
      2   3         <- visible: 3  (3 hides 2)
       \    \
        5    4      <- visible: 4  (4 hides 5)

Right side view = [1, 3, 4]

BFS Queue States:
┌──────────────────────────────────────────────────┐
│ Start      │ [1]                                 │
│ After L0   │ [2, 3]   take last → 1              │
│ After L1   │ [5, 4]   take last → 3              │
│ After L2   │ []       take last → 4              │
└──────────────────────────────────────────────────┘

Trick: after draining a level, level[-1] is the answer.
```

## When To Use This Pattern

- When you see "right side view" or "leftmost/rightmost per level"
  → BFS + take last (or first) element of each level.
- When only one representative per depth is needed
  → collect the whole level, then index it.
- When DFS is tempting but order within a level matters
  → prefer BFS so left-to-right is automatic.
- When you need to find the "edge" node at each tier of a tree
  → level-size BFS with a post-level pick.

## The Approach

Run the same level-order BFS from LC 102.
After draining each level into a temporary list, append only the
last element of that list to the result.
The level list is discarded; only one value per level is kept.
Return the result after BFS completes.

In [ ]:
from collections import deque        # BFS queue
from typing import Optional, List    # type hints


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def make_tree(vals):
    """Build a binary tree from a level-order list.
    None in the list means no node at that position.
    """
    if not vals:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root

In [ ]:
def test_harness(func):
    """Run test cases against func(root) -> List[int]."""
    cases = [
        # (vals_list, expected)
        ([1, 2, 3, None, 5, None, 4],
         [1, 3, 4]),
        ([1, None, 3],
         [1, 3]),                    # only right child
        ([],
         []),                        # empty tree
        ([1],
         [1]),                       # single node
        ([1, 2, 3, 4, 5, 6, 7],
         [1, 3, 7]),                 # full tree — always rightmost
        ([1, 2, None, 3],
         [1, 2, 3]),                 # left-skewed
    ]
    passed = 0
    for vals, expected in cases:
        root = make_tree(vals)
        result = func(root)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(f"{status} | input={vals}")
            print(f"        expected={expected}")
            print(f"        got     ={result}")
    print(f"\n{passed}/{len(cases)} tests passed.")

In [ ]:
def right_side_view(root: Optional[TreeNode]) -> List[int]:
    """
    Return the right side view of a binary tree.

    Args:
        root: Root of the binary tree (may be None).

    Returns:
        List of values visible from the right side,
        one per level, top to bottom.

    Approach:
        BFS level-order. After each level, record level[-1].
    """
    pass


# --- Debug prints (remove pass above before running) ---

# Standard example
t1 = make_tree([1, 2, 3, None, 5, None, 4])
print(right_side_view(t1))   # expected [1, 3, 4]

# Only right child at level 1
t2 = make_tree([1, None, 3])
print(right_side_view(t2))   # expected [1, 3]

# Empty
print(right_side_view(None)) # expected []

# Full tree — always rightmost child wins
t4 = make_tree([1, 2, 3, 4, 5, 6, 7])
print(right_side_view(t4))   # expected [1, 3, 7]

# Left-skewed — left side is the only side
t5 = make_tree([1, 2, None, 3])
print(right_side_view(t5))   # expected [1, 2, 3]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(right_side_view)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force (DFS, track depth + overwrite) | O(n) | O(h) |
| Optimal BFS (level_size + last element)    | O(n) | O(n) |

- **Time O(n):** every node visited once.
- **Space O(n):** deque holds up to n/2 nodes at the widest level.
- DFS uses O(h) stack space (h = height) but is harder to reason
  about correctness for the "rightmost per level" requirement.

## Real World Connection

At **Citi**, a risk dashboard might show only the highest-severity
alert at each tier of a product hierarchy — the "right-side view"
of a risk tree where severity increases to the right.
In **AWS CloudFormation**, when a stack has nested stacks, the
rightmost (latest-deployed) resource at each dependency layer is
the one surfaced in the deployment log summary.
In **data engineering**, a pipeline lineage graph rendered for
stakeholders often shows only the terminal output dataset at each
stage — one representative per layer, exactly this pattern.
Any UI that needs a "summary spine" of a tree structure — one
item per depth — uses right (or left) side view logic.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra